# Clinical Application Layer — Per-Organ Metrics, Anomaly Detection & Compte-Rendu
**Projet : Segmentation sémantique d'images médicales — assistance au diagnostic radiologique**  
**Étudiant : Alaaeddine Bouchamla — Master 1 Génie Télécom, ENISO**  
**Partenaire clinique : Centre de Radiologie Émilie (CRE), Libreville, Gabon**

---

This notebook is the **clinical application layer** that sits on top of the Stage 1 segmentation
models from `03_unet_two_stage.ipynb`. It takes a CRE DICOM volume, runs the appropriate
pretrained model, and produces:

1. **Per-organ volumetric metrics** — volume (mL), mean HU/intensity, presence/absence flag  
2. **Anomaly comparison** — organ metrics vs published normal ranges  
3. **Image-level anomaly score** — convolutional autoencoder reconstruction error  
4. **Annotated overlay images** — colour-coded segmentation overlaid on source slices  
5. **Automated French compte-rendu** — PDF radiology report via ReportLab  

**Prerequisite**: Run `03_unet_two_stage.ipynb` first to load `ct_network` and `mri_network`.
This notebook assumes those variables are in scope (or re-loads the bundles if not).

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q pydicom pyjpegls reportlab Pillow nibabel "monai[all]"

In [ ]:
# ── 2. Imports + paths ─────────────────────────────────────────────────────
import os, io, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import pydicom
from pathlib import Path
from PIL import Image

WORKING_DIR  = '/kaggle/working'
BUNDLE_DIR   = f'{WORKING_DIR}/bundles'
OUTPUT_DIR   = f'{WORKING_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── 3. Load Stage 1 models (re-loads bundles if not already in memory) ────
# If running standalone (not after notebook 03), this cell re-loads the bundles.

from monai.bundle import ConfigParser, download

def load_bundle(bundle_name, bundle_dir, device, highres=False):
    bundle_path = Path(bundle_dir) / bundle_name
    if not bundle_path.exists():
        print(f'Downloading {bundle_name}...')
        download(name=bundle_name, bundle_dir=bundle_dir)
    parser = ConfigParser()
    parser.read_config(str(bundle_path / 'configs' / 'inference.json'))
    parser.read_meta(str(bundle_path / 'configs' / 'metadata.json'))
    parser['bundle_root'] = str(bundle_path)
    if 'highres' in parser:
        parser['highres'] = highres
    network = parser.get_parsed_content('network').to(device)
    model_files = list((bundle_path / 'models').glob('*.pt'))
    ckpt = torch.load(str(model_files[0]), map_location=device, weights_only=False)
    state = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
    network.load_state_dict(state, strict=False)
    network.eval()
    return network

try:
    ct_network
    print('ct_network already loaded.')
except NameError:
    ct_network = load_bundle('wholeBody_ct_segmentation', BUNDLE_DIR, DEVICE)
    print('ct_network loaded from bundle.')

try:
    mri_network
    print('mri_network already loaded.')
except NameError:
    mri_network = load_bundle('wholeBrainSeg_Large_UNEST_segmentation', BUNDLE_DIR, DEVICE)
    print('mri_network loaded from bundle.')

---
## Step 1 — Load and preprocess a CRE DICOM volume

Upload a DICOM ZIP (same format as the CRE backend expects) or point to a folder.

In [ ]:
# ── 4. DICOM loader ────────────────────────────────────────────────────────
# Upload a DICOM ZIP via the Files panel or set DICOM_ZIP_PATH manually.

from google.colab import files as colab_files

DICOM_ZIP_PATH = ''  # set manually if already uploaded, e.g. '/kaggle/working/patient.zip'

if not DICOM_ZIP_PATH:
    print('Upload a DICOM ZIP file:')
    uploaded = colab_files.upload()
    DICOM_ZIP_PATH = list(uploaded.keys())[0]

print(f'Loading: {DICOM_ZIP_PATH}')


def load_dicom_zip(zip_path: str):
    """Extract all DICOM slices from a ZIP, return sorted volume and metadata."""
    datasets = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith('/') or name.startswith('__'):
                continue
            with zf.open(name) as f:
                try:
                    ds = pydicom.dcmread(io.BytesIO(f.read()), force=True)
                    if hasattr(ds, 'pixel_array'):
                        datasets.append(ds)
                except Exception:
                    pass

    # Sort slices by InstanceNumber or ImagePositionPatient
    try:
        datasets.sort(key=lambda d: float(d.ImagePositionPatient[2]))
    except Exception:
        try:
            datasets.sort(key=lambda d: int(d.InstanceNumber))
        except Exception:
            pass

    slices = []
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        # Apply rescale slope/intercept (HU for CT)
        slope = float(getattr(ds, 'RescaleSlope', 1))
        intercept = float(getattr(ds, 'RescaleIntercept', 0))
        arr = arr * slope + intercept
        slices.append(arr)

    volume = np.stack(slices, axis=0)  # (D, H, W)

    # Extract key metadata
    ds0 = datasets[0]
    spacing = [
        float(getattr(ds0, 'SliceThickness', 1.0)),
        float(getattr(ds0, 'PixelSpacing', [1.0, 1.0])[0]),
        float(getattr(ds0, 'PixelSpacing', [1.0, 1.0])[1]),
    ]  # (dz, dy, dx) in mm
    metadata = {
        'Modality':          str(getattr(ds0, 'Modality', 'CT')),
        'BodyPartExamined':  str(getattr(ds0, 'BodyPartExamined', '')),
        'PatientName':       str(getattr(ds0, 'PatientName', 'Anonyme')),
        'PatientAge':        str(getattr(ds0, 'PatientAge', '')),
        'PatientSex':        str(getattr(ds0, 'PatientSex', '')),
        'StudyDescription':  str(getattr(ds0, 'StudyDescription', '')),
        'ProtocolName':      str(getattr(ds0, 'ProtocolName', '')),
        'spacing_mm':        spacing,
    }
    return volume, metadata


volume, metadata = load_dicom_zip(DICOM_ZIP_PATH)
print(f'Volume shape : {volume.shape}  (D x H x W)')
print(f'Spacing (mm) : {metadata["spacing_mm"]}')
print(f'Modality     : {metadata["Modality"]}')
print(f'Body part    : {metadata["BodyPartExamined"]}')
print(f'HU range     : [{volume.min():.0f}, {volume.max():.0f}]')

---
## Step 2 — Segmentation inference (Stage 1 model)

In [ ]:
# ── 5. Modality routing + 2D slice-by-slice inference ─────────────────────
# Full 3D inference via the MONAI bundle transform pipeline is ideal but
# requires large VRAM. For demonstration we run 2D slice-by-slice on a
# representative subset, which fits a free T4.

from monai.transforms import (
    Compose, NormalizeIntensity, Resize, ToTensor
)

def detect_modality(volume, metadata):
    mod = metadata.get('Modality', '').upper()
    body = metadata.get('BodyPartExamined', '').upper()
    if mod == 'CT':
        return 'CT'
    if mod in ('MR', 'MRI'):
        if any(k in body for k in ['BRAIN', 'HEAD', 'CEREBRAL']):
            return 'MRI_BRAIN'
        if any(k in body for k in ['KNEE', 'SHOULDER', 'ELBOW', 'SPINE',
                                     'CERVICAL', 'GENOU', 'EPAULE', 'COUDE', 'RACHIS']):
            return 'MRI_MSK'
        return 'MRI_BRAIN'
    # Intensity heuristic
    return 'CT' if volume.min() < -200 else 'UNKNOWN'


@torch.no_grad()
def run_inference_2d(network, volume, num_classes, n_slices=20):
    """Run 2D inference on evenly-spaced slices. Returns mask volume (D, H, W)."""
    D, H, W = volume.shape
    indices  = np.linspace(0, D - 1, n_slices, dtype=int)
    mask     = np.zeros((D, H, W), dtype=np.uint8)

    for i in indices:
        sl = volume[i].astype(np.float32)
        # Normalise to [0, 1]
        sl = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        t  = torch.from_numpy(sl).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,H,W)
        # Resize to 512x512 if needed
        if H != 512 or W != 512:
            t = torch.nn.functional.interpolate(t, size=(512, 512), mode='bilinear', align_corners=False)
        # Networks expect specific channel count — replicate for multi-channel models
        if network.inc.net[0].in_channels > 1:  # e.g. 4-channel BraTS model
            t = t.repeat(1, network.inc.net[0].in_channels, 1, 1)
        logits = network(t)
        pred   = logits.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)
        if H != 512 or W != 512:
            pred = np.array(Image.fromarray(pred).resize((W, H), Image.NEAREST))
        mask[i] = pred

    return mask, indices


modality = detect_modality(volume, metadata)
print(f'Detected modality: {modality}')

if modality == 'CT':
    network    = ct_network
    NUM_CLASSES = 105  # background + 104 organs
elif modality == 'MRI_BRAIN':
    network    = mri_network
    NUM_CLASSES = 134  # background + 133 structures
elif modality == 'MRI_MSK':
    print('MSK MRI — no Stage 1 model available. Using MedSAM fallback in backend.')
    network = None
    NUM_CLASSES = 2
else:
    print('Unknown modality — defaulting to CT model.')
    network = ct_network
    NUM_CLASSES = 105

if network is not None:
    print('Running 2D inference on representative slices...')
    seg_mask, inferred_indices = run_inference_2d(network, volume, NUM_CLASSES, n_slices=20)
    print(f'Segmentation complete. Mask shape: {seg_mask.shape}')
    print(f'Classes present: {np.unique(seg_mask).tolist()}')

---
## Step 3 — Per-organ volumetric metrics

In [ ]:
# ── 6. Per-organ metrics: volume (mL), mean HU, presence/absence ───────────
# Normal ranges sourced from: Molina et al., 2021 (liver/spleen/kidney volumes);
# Wasserthal et al., 2022 (TotalSegmentator label map).

# TotalSegmentator label map (subset — labels most relevant to CRE case mix)
CT_LABEL_NAMES = {
    1:  'spleen',          2:  'kidney_right',    3:  'kidney_left',
    4:  'gallbladder',     5:  'liver',            6:  'stomach',
    7:  'aorta',           8:  'inferior_vena_cava', 9: 'portal_splenic_vein',
    10: 'pancreas',        11: 'adrenal_gland_right', 12: 'adrenal_gland_left',
    13: 'lung_upper_lobe_left', 14: 'lung_lower_lobe_left',
    15: 'lung_upper_lobe_right', 16: 'lung_middle_lobe_right',
    17: 'lung_lower_lobe_right', 18: 'vertebrae_L5', 19: 'vertebrae_L4',
    20: 'vertebrae_L3',    21: 'vertebrae_L2',     22: 'vertebrae_L1',
}

# Published normal ranges (mean ± 2SD in mL) — used for anomaly flagging
NORMAL_VOLUME_ML = {
    'liver':         (1000, 1800),
    'spleen':        (100,  300),
    'kidney_right':  (110,  200),
    'kidney_left':   (110,  200),
    'pancreas':      (50,   120),
    'gallbladder':   (10,   80),
    'aorta':         (None, None),
}


def compute_organ_metrics(seg_mask, volume_hu, spacing_mm, label_names):
    """Compute volume (mL) and mean HU per organ class."""
    dz, dy, dx = spacing_mm
    voxel_vol_mL = (dz * dy * dx) / 1000.0  # mm³ → mL

    results = []
    present_labels = np.unique(seg_mask)

    for label_id, name in label_names.items():
        if label_id not in present_labels:
            results.append({
                'label_id': label_id, 'name': name,
                'present': False, 'volume_mL': 0.0, 'mean_hu': None,
                'anomaly': False, 'anomaly_reason': ''
            })
            continue

        voxels   = seg_mask == label_id
        vol_mL   = float(voxels.sum()) * voxel_vol_mL
        mean_hu  = float(volume_hu[voxels].mean()) if voxels.sum() > 0 else None

        # Anomaly check against normal range
        anomaly = False
        reason  = ''
        if name in NORMAL_VOLUME_ML:
            lo, hi = NORMAL_VOLUME_ML[name]
            if lo is not None and vol_mL < lo:
                anomaly = True; reason = f'volume {vol_mL:.0f} mL < normal min {lo} mL'
            elif hi is not None and vol_mL > hi:
                anomaly = True; reason = f'volume {vol_mL:.0f} mL > normal max {hi} mL'

        results.append({
            'label_id': label_id, 'name': name,
            'present': True, 'volume_mL': vol_mL, 'mean_hu': mean_hu,
            'anomaly': anomaly, 'anomaly_reason': reason
        })

    return results


label_names = CT_LABEL_NAMES if modality == 'CT' else {}
organ_metrics = compute_organ_metrics(seg_mask, volume, metadata['spacing_mm'], label_names)

print(f'{'Organ':<30} {'Vol (mL)':>10} {'Mean HU':>10} {'Anomaly'}')
print('-' * 65)
for m in organ_metrics:
    if not m['present']:
        continue
    flag = '  *** ANOMALIE ***' if m['anomaly'] else ''
    hu   = f"{m['mean_hu']:.1f}" if m['mean_hu'] is not None else 'N/A'
    print(f"{m['name']:<30} {m['volume_mL']:>10.1f} {hu:>10} {flag}")

---
## Step 4 — Convolutional Autoencoder: Image-Level Anomaly Score

In [ ]:
# ── 7. Convolutional autoencoder for anomaly detection ─────────────────────
# Trained on normal-appearing slices. High reconstruction error → potential anomaly.
# Here we define and run it untrained (demo mode). For real use, train it on
# normal CRE slices and save weights to AUTOENCODER_WEIGHTS_PATH.

AUTOENCODER_WEIGHTS_PATH = f'{WORKING_DIR}/autoencoder.pth'  # set if trained


class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


autoencoder = ConvAutoencoder().to(DEVICE)
if Path(AUTOENCODER_WEIGHTS_PATH).exists():
    autoencoder.load_state_dict(torch.load(AUTOENCODER_WEIGHTS_PATH,
                                            map_location=DEVICE, weights_only=False))
    autoencoder.eval()
    print('Autoencoder weights loaded.')
else:
    autoencoder.eval()
    print('Autoencoder running in demo mode (untrained). '
          'Train on normal CRE slices and save to AUTOENCODER_WEIGHTS_PATH for real scores.')


@torch.no_grad()
def compute_anomaly_scores(model, volume, n_slices=20):
    """Return per-slice MSE reconstruction error (higher = more anomalous)."""
    D, H, W = volume.shape
    indices  = np.linspace(0, D - 1, n_slices, dtype=int)
    scores   = []
    for i in indices:
        sl = volume[i].astype(np.float32)
        sl = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        t  = torch.from_numpy(sl).unsqueeze(0).unsqueeze(0).to(DEVICE)
        t  = torch.nn.functional.interpolate(t, size=(512, 512), mode='bilinear', align_corners=False)
        recon = model(t)
        mse   = float(((t - recon) ** 2).mean().item())
        scores.append((int(i), mse))
    return scores


anomaly_scores = compute_anomaly_scores(autoencoder, volume)
scores_arr     = np.array([s[1] for s in anomaly_scores])
ANOMALY_THRESHOLD = float(np.mean(scores_arr) + 2 * np.std(scores_arr))

print(f'Anomaly threshold (mean + 2σ): {ANOMALY_THRESHOLD:.6f}')
print(f'Max reconstruction error     : {scores_arr.max():.6f}')
flagged = [(idx, sc) for idx, sc in anomaly_scores if sc > ANOMALY_THRESHOLD]
print(f'Flagged slices               : {len(flagged)}')
for idx, sc in flagged:
    print(f'  Slice {idx:3d}: MSE = {sc:.6f}')

# Plot anomaly scores
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot([s[0] for s in anomaly_scores], scores_arr, color='#1a3c6e')
ax.axhline(ANOMALY_THRESHOLD, color='#c0392b', linestyle='--', label=f'Threshold ({ANOMALY_THRESHOLD:.4f})')
ax.set_xlabel('Slice index'); ax.set_ylabel('Reconstruction MSE')
ax.set_title('Autoencoder Anomaly Scores per Slice')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/anomaly_scores.png', dpi=150)
plt.show()

---
## Step 5 — Annotated overlay images

In [ ]:
# ── 8. Overlay visualisation ───────────────────────────────────────────────
# Colour-coded segmentation overlaid on source slices, saved as PNG.

import matplotlib.cm as cm

OVERLAY_ALPHA = 0.4
N_DISPLAY     = 6   # number of slices to show in the figure

display_indices = [i for i in inferred_indices[::max(1, len(inferred_indices)//N_DISPLAY)][:N_DISPLAY]]

cmap_seg = cm.get_cmap('tab20', max(NUM_CLASSES, 20))

fig, axes = plt.subplots(2, N_DISPLAY // 2, figsize=(18, 8))
axes = axes.flatten()

for ax_i, slice_idx in enumerate(display_indices):
    src = volume[slice_idx].astype(np.float32)
    # Window level for display (CT: soft tissue window -40/400 HU)
    if modality == 'CT':
        src = np.clip(src, -40 - 200, -40 + 200)
    src = (src - src.min()) / (src.max() - src.min() + 1e-8)

    mask_sl = seg_mask[slice_idx]
    colour_mask = cmap_seg(mask_sl / max(NUM_CLASSES - 1, 1))[..., :3]

    blended = src[..., np.newaxis] * (1 - OVERLAY_ALPHA) + colour_mask * OVERLAY_ALPHA
    blended = np.clip(blended, 0, 1)

    axes[ax_i].imshow(blended)
    axes[ax_i].set_title(f'Slice {slice_idx}', fontsize=9)
    axes[ax_i].axis('off')

plt.suptitle(f'Segmentation Overlay — {modality} | {metadata["PatientName"]}', fontsize=12)
plt.tight_layout()
overlay_path = f'{OUTPUT_DIR}/segmentation_overlay.png'
plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {overlay_path}')

---
## Step 6 — Generate French Compte-Rendu (PDF)

In [ ]:
# ── 9. Generate French radiology report (PDF) ──────────────────────────────
# Builds the compte-rendu using ReportLab directly (no backend dependency).

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from datetime import datetime

CLINICAL_INDICATION = 'Bilan de routine'  # set this per exam
PRESCRIBING_DOCTOR  = 'Dr. [à renseigner]'

PDF_PATH = f'{OUTPUT_DIR}/compte_rendu_draft.pdf'

styles = getSampleStyleSheet()
H1  = ParagraphStyle('H1',  fontName='Helvetica-Bold',   fontSize=13, spaceAfter=6)
H2  = ParagraphStyle('H2',  fontName='Helvetica-Bold',   fontSize=10, spaceAfter=4)
Nrm = ParagraphStyle('Nrm', fontName='Helvetica',         fontSize=9,  spaceAfter=3)
Sml = ParagraphStyle('Sml', fontName='Helvetica-Oblique', fontSize=8,  textColor=colors.grey)
Red = ParagraphStyle('Red', fontName='Helvetica-Bold',    fontSize=9,  textColor=colors.red)

modality_label = 'TDM' if modality == 'CT' else 'IRM'
body           = metadata.get('BodyPartExamined', '').upper() or 'CORPS ENTIER'
exam_type      = f'{modality_label} {body}'
date_str       = datetime.now().strftime('%d/%m/%Y')

doc   = SimpleDocTemplate(PDF_PATH, pagesize=A4,
                           leftMargin=2*cm, rightMargin=2*cm,
                           topMargin=2*cm, bottomMargin=2*cm)
story = []

# Header
story.append(Paragraph('CENTRE DE RADIOLOGIE EMILIE — LIBREVILLE, GABON', H1))
story.append(Paragraph('BROUILLON IA — À VALIDER PAR LE RADIOLOGUE', Red))
story.append(Spacer(1, 0.3*cm))

# Patient info table
info = [
    ['Patient', str(metadata['PatientName']),
     'Date', date_str],
    ['Age / Sexe', f"{metadata['PatientAge']} / {metadata['PatientSex']}",
     'Médecin prescripteur', PRESCRIBING_DOCTOR],
    ['Examen', exam_type,
     'Indication', CLINICAL_INDICATION],
]
t = Table(info, colWidths=[3*cm, 6*cm, 4*cm, 5*cm])
t.setStyle(TableStyle([
    ('FONTNAME',    (0,0), (-1,-1), 'Helvetica'),
    ('FONTNAME',    (0,0), (0,-1), 'Helvetica-Bold'),
    ('FONTNAME',    (2,0), (2,-1), 'Helvetica-Bold'),
    ('FONTSIZE',    (0,0), (-1,-1), 8),
    ('GRID',        (0,0), (-1,-1), 0.3, colors.lightgrey),
    ('BACKGROUND',  (0,0), (0,-1), colors.HexColor('#e8eef7')),
    ('BACKGROUND',  (2,0), (2,-1), colors.HexColor('#e8eef7')),
]))
story.append(t)
story.append(Spacer(1, 0.5*cm))

# RÉSULTAT
story.append(Paragraph('RÉSULTAT :', H2))
present_organs = [m for m in organ_metrics if m['present']]
if present_organs:
    for m in present_organs:
        line = f"- {m['name'].replace('_', ' ').title()} : volume {m['volume_mL']:.0f} mL"
        if m['mean_hu'] is not None:
            line += f", densité moyenne {m['mean_hu']:.0f} UH"
        if m['anomaly']:
            line += f" — ANOMALIE : {m['anomaly_reason']}"
            story.append(Paragraph(line, Red))
        else:
            story.append(Paragraph(line, Nrm))
else:
    story.append(Paragraph('- Analyse en cours. Résultats en attente de validation.', Nrm))
story.append(Spacer(1, 0.3*cm))

# CONCLUSION
story.append(Paragraph('CONCLUSION :', H2))
anomalous = [m for m in organ_metrics if m['anomaly']]
if anomalous:
    names = ', '.join(m['name'].replace('_', ' ') for m in anomalous)
    concl = (f"Anomalie(s) volumétrique(s) détectée(s) par le système IA au niveau de : {names}. "
             f"À corréler avec la clinique et à valider par le radiologue.")
else:
    concl = ("Examen d'aspect normal selon l'analyse IA préliminaire. "
             "À confirmer par le radiologue.")
story.append(Paragraph(concl, Nrm))
story.append(Spacer(1, 0.5*cm))
story.append(Paragraph(
    f'Score IA de normalité : {1 - min(scores_arr.max() * 10, 1.0):.0%} '
    f'| Modèle : {"wholeBody_ct_segmentation" if modality=="CT" else "wholeBrainSeg_UNEST"} '
    f'| Ce rapport est un brouillon IA non validé.',
    Sml
))

doc.build(story)
print(f'PDF generated: {PDF_PATH}  ({Path(PDF_PATH).stat().st_size:,} bytes)')

In [ ]:
# ── 10. Download all outputs ───────────────────────────────────────────────
from google.colab import files as colab_files

print('Output files:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

for f in sorted(Path(OUTPUT_DIR).iterdir()):
    colab_files.download(str(f))

print('\nDone. Place compte_rendu_draft.pdf in the backend pdfs/ directory.')
print('Place segmentation_overlay.png in uploads/ for the frontend to serve.')